# 5 · Support Ticket Router
## Classification → Routing → Guarded Draft Reply
⏱ ~20 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/5-support-ticket-router/support_ticket_router_workbook.ipynb)

Support teams receive dozens of ticket types — billing disputes, outages, account changes, feature requests — each needing a different tone, team, and urgency level. Routing them manually wastes time and introduces inconsistency.

This example builds a **two-stage pipeline**:
1. A **classifier** reads the ticket and returns a typed routing decision
2. A **drafter** uses the routing result to pick the right system prompt and draft a first-response email

The `escalate` field acts as a **human-in-the-loop gate** — flagged tickets pause before sending until a human reviews them.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The routing problem — why one prompt can't handle all ticket types |
| 2 | Stage 1: classify with `TicketClassification` |
| 3 | Stage 2: conditional drafting — team-specific system prompts |
| 4 | The escalation gate — human-in-the-loop via a boolean flag |
| 5 | Running the full pipeline end-to-end |
| ★ | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab (run the install cell)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    %pip install -q langchain-openai pydantic python-dotenv
    print("Packages installed.")
else:
    print("Local environment — install with: pip install langchain-openai pydantic python-dotenv")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
    print("API key loaded from .env.")

---

## Part 1 — The Routing Problem

---

A single `"classify this support ticket"` prompt can tell you *what* the ticket is about — but it can't also draft a reply, because the right tone and content differ completely by team:

| Team | Tone | Key actions |
|------|------|-------------|
| **Billing** | Empathetic, formal | Acknowledge dispute, set 1-2 day expectation |
| **Engineering** | Technical, thorough | Ask for reproduction steps, flag outages |
| **Account Management** | Warm, retention-focused | Offer retention path for cancellations |
| **Product** | Appreciative, non-committal | Log feature request, no timeline promises |
| **General Support** | Friendly, concise | Resolve in one reply where possible |

The solution: **two sequential LLM calls**, each with its own typed schema.

```
raw ticket
    │
    ▼
┌─────────────────────┐
│  CLASSIFIER         │   ← with_structured_output(TicketClassification)
│  type, urgency,     │
│  team, confidence   │
└─────────────────────┘
    │  team = "billing"
    ▼
┌─────────────────────┐
│  DRAFTER            │   ← team-specific system prompt
│  subject, body,     │     + with_structured_output(DraftReply)
│  internal_note,     │
│  escalate           │
└─────────────────────┘
```

---

## Part 2 — Stage 1: The Classifier Schema

---

The classifier needs to return four things: what type of ticket it is, how urgent, which team handles it, and how confident the model is. We encode all of this in one Pydantic model.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class TicketClassification(BaseModel):
    ticket_type: Literal["billing", "technical", "account", "feature_request", "other"]
    urgency: Literal["critical", "high", "medium", "low"]
    team: Literal["billing", "engineering", "account_management", "product", "general_support"]
    confidence: float = Field(ge=0.0, le=1.0, description="Classifier confidence 0-1")
    reasoning: str = Field(description="One sentence explaining the routing decision")

# The Literal constraints mean:
# - The model CANNOT return "URGENT" instead of "critical"
# - The model CANNOT return "billing_team" instead of "billing"
# - confidence MUST be between 0 and 1
# If the model tries to return an invalid value, Pydantic raises a ValidationError
# and the LangChain structured output layer retries automatically
print("Schema defined. Literal fields enforce exact allowed values.")

---

## Part 3 — Building the Classifier

---

`with_structured_output()` binds the schema to the LLM. When the model responds, LangChain:
1. Passes the schema as a tool/function definition
2. Forces the model to call that function
3. Validates the result against Pydantic
4. Retries on validation error

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

CLASSIFIER_PROMPT = SystemMessage(
    """You are a customer support ticket classifier.

Given a support ticket, classify it with:
- ticket_type: billing | technical | account | feature_request | other
- urgency:
    critical = service down, data loss, security issue
    high     = major feature broken, billing dispute
    medium   = degraded performance, billing question
    low      = general question, feature request
- team: the team best equipped to handle it
    billing           → payment, invoice, charge issues
    engineering       → bugs, outages, data issues
    account_management → account changes, upgrades, cancellations
    product           → feature requests, feedback
    general_support   → everything else
- confidence: your certainty 0.0–1.0
- reasoning: one sentence explaining your routing decision"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
classifier = CLASSIFIER_PROMPT | llm.with_structured_output(TicketClassification)
print("Classifier ready.")

In [ ]:
# Run the classifier on a billing dispute
billing_ticket = """Subject: Charged twice this month - invoice #4821

Hi, I noticed my credit card was charged $99 twice on June 1st.
Invoice #4821 shows a duplicate charge. Please refund the extra charge ASAP.
This is really frustrating."""

result = classifier.invoke(billing_ticket)
print(f"Type:       {result.ticket_type}")
print(f"Urgency:    {result.urgency}")
print(f"Team:       {result.team}")
print(f"Confidence: {result.confidence:.0%}")
print(f"Reasoning:  {result.reasoning}")
print(f"\nPython type of result: {type(result)}")  # TicketClassification, not a string

---

## Part 4 — Stage 2: Conditional Drafting

---

The key insight: **different teams need different tones**. We solve this by keeping one system prompt per team in a dictionary, then selecting the right one based on the classification result.

This is a simple but powerful pattern — instead of one massive prompt that tries to handle everything, you have focused prompts that each do one thing well.

In [ ]:
class DraftReply(BaseModel):
    subject: str
    body: str
    internal_note: str = Field(description="Note for the support agent, not sent to customer")
    escalate: bool = Field(description="True if this needs a human to review before sending")

DRAFT_PROMPTS = {
    "billing": SystemMessage(
        """You draft first-response emails for the billing support team.
Be empathetic, acknowledge the issue, and set a clear expectation for resolution time (1-2 business days).
Always include: acknowledgment, next steps, and a contact path if urgent.
Set escalate=True for any dispute over $500 or involving a subscription cancellation."""
    ),
    "engineering": SystemMessage(
        """You draft first-response emails for the engineering/technical support team.
Acknowledge the issue, ask for relevant details (OS, browser, steps to reproduce) if not provided.
Set escalate=True for any outage, data loss, or security concern."""
    ),
    "account_management": SystemMessage(
        """You draft first-response emails for the account management team.
Be warm and professional. For cancellation requests, acknowledge and offer a retention path.
Set escalate=True for enterprise accounts or churn risk."""
    ),
    "product": SystemMessage(
        """You draft first-response emails for the product team.
Thank the customer for feedback, confirm it's been logged, set expectations (no timeline commitments).
escalate=False unless the feature request is blocking product use."""
    ),
    "general_support": SystemMessage(
        """You draft first-response emails for general support.
Be helpful and concise. Aim to resolve in one reply where possible.
Set escalate=True only if the issue requires account access or billing changes."""
    ),
}

print(f"Draft prompts defined for {len(DRAFT_PROMPTS)} teams: {list(DRAFT_PROMPTS.keys())}")

---

## Part 5 — The Escalation Gate

---

The `escalate` field is the human-in-the-loop mechanism. Before any draft is sent, the pipeline checks this flag:

```python
if draft.escalate:
    # pause → route to human review queue
else:
    # auto-send or send with one-click confirmation
```

This is a minimal but effective pattern for keeping humans in the loop on high-risk tickets without slowing down routine ones. The LLM decides what's risky; humans only see what needs attention.

In [ ]:
DRAFT_CONTEXT = """
You are replying to this customer support ticket:

Subject: {subject}
From: {customer_name} <{customer_email}>

---
{body}
---

Classification: {ticket_type} / {urgency} urgency → routed to {team}
"""

def run_pipeline(ticket: dict) -> dict:
    """
    ticket: {subject, body, customer_name, customer_email}
    returns: {classification, draft}
    """
    clf_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    classification = (CLASSIFIER_PROMPT | clf_llm.with_structured_output(TicketClassification)).invoke(
        f"Subject: {ticket['subject']}\n\n{ticket['body']}"
    )

    team = classification.team
    system = DRAFT_PROMPTS.get(team, DRAFT_PROMPTS["general_support"])
    draft_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    drafter = system | draft_llm.with_structured_output(DraftReply)

    context = DRAFT_CONTEXT.format(
        subject=ticket["subject"],
        customer_name=ticket["customer_name"],
        customer_email=ticket["customer_email"],
        body=ticket["body"],
        ticket_type=classification.ticket_type,
        urgency=classification.urgency,
        team=team,
    )
    draft = drafter.invoke(context)

    return {"classification": classification, "draft": draft}

print("Pipeline function defined.")

In [ ]:
SAMPLE_TICKETS = [
    {
        "subject": "Charged twice this month - invoice #4821",
        "body": "Hi, I noticed my credit card was charged $99 twice on June 1st. Invoice #4821 shows a duplicate charge. Please refund the extra charge ASAP. This is really frustrating.",
        "customer_name": "Sarah Chen",
        "customer_email": "schen@example.com",
    },
    {
        "subject": "Dashboard not loading - production down",
        "body": "Our entire team cannot access the dashboard since 9 AM EST. We're getting a 502 error. This is blocking all our work. We're on the Enterprise plan.",
        "customer_name": "Marcus Torres",
        "customer_email": "m.torres@bigcorp.com",
    },
    {
        "subject": "How do I add team members?",
        "body": "Hi, I'm trying to invite my colleagues to our workspace but I can't find where to do it. Can you help?",
        "customer_name": "Priya Patel",
        "customer_email": "priya@startup.io",
    },
]

for ticket in SAMPLE_TICKETS:
    print(f"\n{'='*60}")
    print(f"TICKET: {ticket['subject']}")
    print(f"FROM:   {ticket['customer_name']}")
    print("=" * 60)

    result = run_pipeline(ticket)
    clf = result["classification"]
    draft = result["draft"]

    print("\n[CLASSIFICATION]")
    print(f"  Type:       {clf.ticket_type}")
    print(f"  Urgency:    {clf.urgency}")
    print(f"  Team:       {clf.team}")
    print(f"  Confidence: {clf.confidence:.0%}")
    print(f"  Reasoning:  {clf.reasoning}")

    print("\n[DRAFT REPLY]")
    print(f"  Subject:  {draft.subject}")
    print(f"  Escalate: {'YES ⚠ — needs human review' if draft.escalate else 'No — safe to send'}")
    print("  Body:\n")
    for line in draft.body.split("\n"):
        print(f"    {line}")
    print(f"\n  [Internal note]: {draft.internal_note}")

---

## Exercise 1 — Add a `sentiment` field

Add a `sentiment: Literal["frustrated", "neutral", "positive"]` field to `TicketClassification`.
Re-run the classifier on all three sample tickets and verify it detects Sarah's frustration.

**Hint:** Add the field to the schema and update the classifier prompt to describe when each value applies.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Add sentiment to TicketClassification, update the prompt, re-run.

from typing import Literal
from pydantic import BaseModel, Field

class TicketClassificationEx1(BaseModel):
    ticket_type: Literal["billing", "technical", "account", "feature_request", "other"]
    urgency: Literal["critical", "high", "medium", "low"]
    team: Literal["billing", "engineering", "account_management", "product", "general_support"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    # TODO: add sentiment field here

# TODO: update the system prompt to describe sentiment values
# TODO: rebuild classifier and test on SAMPLE_TICKETS

In [ ]:
# ── Exercise 1 Answer Key ─────────────────────────────────────────────────────
class TicketClassificationV2(BaseModel):
    ticket_type: Literal["billing", "technical", "account", "feature_request", "other"]
    urgency: Literal["critical", "high", "medium", "low"]
    team: Literal["billing", "engineering", "account_management", "product", "general_support"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    sentiment: Literal["frustrated", "neutral", "positive"] = Field(
        description="Customer emotional tone: frustrated=complaints/urgency, positive=praise/thanks, neutral=questions"
    )

CLASSIFIER_V2_PROMPT = SystemMessage(
    """You are a customer support ticket classifier.
Classify tickets with ticket_type, urgency, team, confidence, reasoning, and sentiment.
sentiment: frustrated = complaints or urgency, positive = praise or thanks, neutral = questions/requests."""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
clf_v2 = CLASSIFIER_V2_PROMPT | llm.with_structured_output(TicketClassificationV2)

for ticket in SAMPLE_TICKETS:
    r = clf_v2.invoke(f"Subject: {ticket['subject']}\n\n{ticket['body']}")
    print(f"{ticket['customer_name']:15} → {r.sentiment:12} | {r.urgency:8} | {r.team}")

---

## Exercise 2 — Add a `language` routing field

Some support teams handle tickets in specific languages.
Add a `language: Literal["en", "es", "fr", "other"]` field to `TicketClassification`
and add a Spanish test ticket.

```python
spanish_ticket = {
    "subject": "No puedo acceder a mi cuenta",
    "body": "Buenos días, perdí acceso a mi cuenta después de cambiar mi contraseña. ¿Pueden ayudarme?",
    "customer_name": "Carlos García",
    "customer_email": "cgarcia@empresa.mx",
}
```

Verify the classifier correctly identifies `language="es"` and routes to `account_management`.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
spanish_ticket = {
    "subject": "No puedo acceder a mi cuenta",
    "body": "Buenos días, perdí acceso a mi cuenta después de cambiar mi contraseña. ¿Pueden ayudarme?",
    "customer_name": "Carlos García",
    "customer_email": "cgarcia@empresa.mx",
}

class TicketClassificationEx2(BaseModel):
    ticket_type: Literal["billing", "technical", "account", "feature_request", "other"]
    urgency: Literal["critical", "high", "medium", "low"]
    team: Literal["billing", "engineering", "account_management", "product", "general_support"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    # TODO: add language field

# TODO: rebuild classifier, run on spanish_ticket

In [ ]:
# ── Exercise 2 Answer Key ─────────────────────────────────────────────────────
class TicketClassificationV3(BaseModel):
    ticket_type: Literal["billing", "technical", "account", "feature_request", "other"]
    urgency: Literal["critical", "high", "medium", "low"]
    team: Literal["billing", "engineering", "account_management", "product", "general_support"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    language: Literal["en", "es", "fr", "other"] = Field(
        description="Primary language of the ticket"
    )

CLASSIFIER_V3_PROMPT = SystemMessage(
    """You are a multilingual customer support ticket classifier.
Classify tickets with ticket_type, urgency, team, confidence, reasoning, and language.
language: en=English, es=Spanish, fr=French, other=anything else."""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
clf_v3 = CLASSIFIER_V3_PROMPT | llm.with_structured_output(TicketClassificationV3)

r = clf_v3.invoke(f"Subject: {spanish_ticket['subject']}\n\n{spanish_ticket['body']}")
print(f"Type:     {r.ticket_type}")
print(f"Urgency:  {r.urgency}")
print(f"Team:     {r.team}")
print(f"Language: {r.language}")
print(f"Reasoning: {r.reasoning}")

---

## What's Next?

You've built a two-stage routing pipeline with a human-in-the-loop escalation gate.

**Key patterns used here:**
- Sequential LLM calls — classifier output feeds into drafter input
- Conditional system prompts — team-specific context via a dict lookup
- `escalate` flag — typed boolean as a human-review gate

**Next examples in this series:**
- **6 — Resume Screener** — score many inputs against a rubric (comparable structured output)
- **7 — RFP Parser** — extract structured data from long documents
- **8 — Multi-Agent Research** — supervisor delegates to specialist sub-agents

---
*Example 5 of 8 · agent-use-cases*